# 2026 World Cup Predictor

This notebook trains a calibrated 1X2 match-outcome model and uses the predictions to simulate the 2026 World Cup tournament.


In [2]:
# Optional setup cell: installs missing packages in fresh environments.
# Run this once if imports fail. It is safe to leave here for Colab / fresh environments.

import importlib.util
import subprocess
import sys

required_packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "scikit-learn",
    "xgboost": "xgboost",
}

missing = [pip_name for import_name, pip_name in required_packages.items()
           if importlib.util.find_spec(import_name) is None]

if missing:
    print("Installing missing packages into:", sys.executable)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required packages are available in:", sys.executable)

All required packages are available in: /opt/anaconda3/bin/python


## 1. Imports and configuration

In [3]:
# Import project classes and set the input/output paths.
from pathlib import Path

import pandas as pd

from predictor_pipeline_final_v2 import (
    OFFICIAL_GROUPS_2026,
    ProjectConfig,
    WorldCupPredictorProject,
    ExpectedValueCalculator,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

DATA_PATH = Path("df_fin_v1.csv")
OUTPUT_DIR = Path("final_outputs_v2")

In [4]:
# Check the dataset and show basic size information before modelling.
# Basic file check.
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Keep df_fin_v1.csv in the same folder as this notebook, "
        "or change DATA_PATH to the correct path."
    )

raw_df = pd.read_csv(DATA_PATH)
print("Rows:", len(raw_df))
print("Columns:", len(raw_df.columns))
print("Future/unplayed matches:", raw_df["result"].isna().sum())
raw_df.tail(3)

Rows: 3143
Columns: 23
Future/unplayed matches: 72


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,home_rank,away_rank,rank_difference,home_avg_age,home_market_value,away_avg_age,away_market_value,age_difference,value_difference,result,home_form,away_form,form_difference
3140,2026-06-27,Congo DR,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,USA,True,2026,46,50,-4,29.1,143900000.0,28.5,8.533000e+07,0.6,5.857000e+07,NaN,10,11,-1
3141,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,USA,True,2026,34,4,30,30.4,34550000.0,27.2,1.360000e+09,3.2,-1.325450e+09,NaN,11,15,-4
3142,2026-06-27,Croatia,Ghana,NaN,NaN,FIFA World Cup,Philadelphia,USA,True,2026,11,73,-62,28.4,387300000.0,26.8,2.346000e+08,1.6,1.527000e+08,NaN,13,10,3


## 2. Group dictionary

The simulation uses an explicit group dictionary. This keeps group assignment separate from the fixture data.


In [5]:
# Display the group structure used by the simulator.
OFFICIAL_GROUPS_2026

{'A': ['Mexico', 'South Africa', 'Korea Republic', 'Czechia'],
 'B': ['Canada', 'Qatar', 'Switzerland', 'Bosnia and Herzegovina'],
 'C': ['Brazil', 'Morocco', 'Haiti', 'Scotland'],
 'D': ['USA', 'Paraguay', 'Australia', 'Turkey'],
 'E': ['Germany', 'Curaçao', "Côte d'Ivoire", 'Ecuador'],
 'F': ['Netherlands', 'Japan', 'Sweden', 'Tunisia'],
 'G': ['Belgium', 'Egypt', 'IR Iran', 'New Zealand'],
 'H': ['Spain', 'Cabo Verde', 'Saudi Arabia', 'Uruguay'],
 'I': ['France', 'Senegal', 'Iraq', 'Norway'],
 'J': ['Argentina', 'Algeria', 'Austria', 'Jordan'],
 'K': ['Portugal', 'Congo DR', 'Uzbekistan', 'Colombia'],
 'L': ['England', 'Croatia', 'Ghana', 'Panama']}

## 3. Run full model + tournament pipeline

In [6]:
# Run the complete workflow: data preparation, training, prediction and simulation.
config = ProjectConfig(
    data_path=DATA_PATH,
    output_dir=OUTPUT_DIR,
)

project = WorldCupPredictorProject(config)
outputs = project.run()

print("Pipeline finished.")
print("Output folder:", OUTPUT_DIR.resolve())

Pipeline finished.
Output folder: /Users/lukaspodhorsky/Downloads/predictor_project_final_v2/final_outputs_v2


In [7]:
# Show basic model performance on the held-out test matches.
evaluation = outputs["evaluation"]
print("Test matches:", evaluation["test_matches"])
print("Accuracy:", round(evaluation["accuracy"], 4))
print("Log loss:", round(evaluation["log_loss"], 4))

Test matches: 591
Accuracy: 0.6565
Log loss: 0.7932


## 4. Future match probabilities

In [8]:
# Show model probabilities for future group-stage fixtures.
future_predictions = outputs["future_predictions"]
future_predictions[[
    "date", "home_team", "away_team", "pred_label",
    "prob_home_win", "prob_draw", "prob_away_win"
]].head(12)

,date,home_team,away_team,pred_label,prob_home_win,prob_draw,prob_away_win
0,2026-06-11,Mexico,South Africa,Home win,0.715319,0.199415,0.085267
1,2026-06-11,Korea Republic,Czechia,Draw,0.316410,0.434490,0.249100
2,2026-06-12,Canada,Bosnia and Herzegovina,Home win,0.670114,0.222249,0.107637
3,2026-06-12,USA,Paraguay,Home win,0.676002,0.221609,0.102390
4,2026-06-13,Australia,Turkey,Draw,0.313645,0.358490,0.327865
5,2026-06-13,Brazil,Morocco,Home win,0.463783,0.306902,0.229315
6,2026-06-13,Qatar,Switzerland,Away win,0.158857,0.323081,0.518062
7,2026-06-13,Haiti,Scotland,Away win,0.146877,0.259014,0.594109
8,2026-06-14,Sweden,Tunisia,Home win,0.434449,0.315019,0.250533
9,2026-06-14,Netherlands,Japan,Home win,0.530221,0.302180,0.167599


## 5. Group stage tables

In [9]:
# Show simulated group tables after predicted group results.
group_table = outputs["group_table"]
group_table.head(16)

,team,group,points,wins,draws,losses,expected_points,fifa_rank,h2h_bonus,group_position
0,Mexico,A,9,3,0,0,6.707459,15,0.0,1
1,Korea Republic,A,4,1,1,1,4.003753,25,0.0,2
2,Czechia,A,4,1,1,1,3.333602,40,0.0,3
3,South Africa,A,0,0,0,3,2.135244,60,0.0,4
4,Switzerland,B,9,3,0,0,5.886429,19,0.0,1
5,Canada,B,6,2,0,1,5.504204,27,0.0,2
6,Bosnia and Herzegovina,B,3,1,0,2,2.601360,64,0.0,3
7,Qatar,B,0,0,0,3,2.490091,56,0.0,4
8,Brazil,C,9,3,0,0,5.786367,5,0.0,1
9,Morocco,C,6,2,0,1,5.169641,7,0.0,2


## 6. Best third-place assignments

The top third-place teams are ranked and assigned to available Round-of-32 slots.


In [10]:
# Show how best third-place teams are placed into Round-of-32 slots.
outputs["third_place_assignments"]

,match_number,third_place_group,team
0,74,A,Czechia
1,77,D,Paraguay
2,79,E,Ecuador
3,80,I,Senegal
4,81,F,Sweden
5,82,J,Austria
6,85,G,IR Iran
7,87,L,Panama


## 7. Knockout pathway


In [11]:
# Show the full simulated knockout bracket.
knockout_bracket = outputs["knockout_bracket"]
knockout_bracket

,round,match_number,home_team,away_team,prob_home_win_90,prob_draw_90,prob_away_win_90,winner,loser,advancement_rule
0,Round of 32,73,Korea Republic,Canada,0.301114,0.380407,0.318480,Canada,Korea Republic,"90min winner; if draw, higher baseline win pro..."
1,Round of 32,74,Germany,Czechia,0.718719,0.174016,0.107265,Germany,Czechia,"90min winner; if draw, higher baseline win pro..."
2,Round of 32,75,Netherlands,Morocco,0.496255,0.287586,0.216159,Netherlands,Morocco,"90min winner; if draw, higher baseline win pro..."
3,Round of 32,76,Brazil,Japan,0.611508,0.206292,0.182200,Brazil,Japan,"90min winner; if draw, higher baseline win pro..."
4,Round of 32,77,France,Paraguay,0.742061,0.157120,0.100819,France,Paraguay,"90min winner; if draw, higher baseline win pro..."
5,Round of 32,78,Côte d'Ivoire,Norway,0.300002,0.337540,0.362458,Norway,Côte d'Ivoire,"90min winner; if draw, higher baseline win pro..."
6,Round of 32,79,Mexico,Ecuador,0.298655,0.455944,0.245401,Mexico,Ecuador,"90min winner; if draw, higher baseline win pro..."
7,Round of 32,80,England,Senegal,0.678180,0.175845,0.145975,England,Senegal,"90min winner; if draw, higher baseline win pro..."
8,Round of 32,81,USA,Sweden,0.325975,0.438702,0.235323,USA,Sweden,"90min winner; if draw, higher baseline win pro..."
9,Round of 32,82,Belgium,Austria,0.485920,0.358338,0.155742,Belgium,Austria,"90min winner; if draw, higher baseline win pro..."


In [12]:
# Extract the final winner and third-place winner from the bracket table.
champion = knockout_bracket.loc[knockout_bracket["round"].eq("Final"), "winner"].iloc[0]
third_place = knockout_bracket.loc[knockout_bracket["round"].eq("Third-place play-off"), "winner"].iloc[0]
print("Predicted champion:", champion)
print("Predicted third place:", third_place)

Predicted champion: Spain
Predicted third place: Brazil


## 8. Bet Builder example

In [13]:
# Example Bet Builder calculation using model probability and decimal odds.
# Example: evaluate whether a hypothetical decimal odd is +EV.
# Replace probability and odds with the user's selected match/outcome.

probability = float(future_predictions.iloc[0]["prob_home_win"])
decimal_odds = 2.10

ExpectedValueCalculator.evaluate_selection(probability, decimal_odds)

{'model_probability': 0.7153186054091446,
 'bookmaker_implied_probability': 0.47619047619047616,
 'expected_value': 0.5021690713592037,
 'is_positive_ev': True}

## 9. Files written

The pipeline writes these CSV outputs to `final_outputs_v2/`:

- `evaluation_summary.csv`
- `test_predictions.csv`
- `future_predictions.csv`
- `group_matches.csv`
- `group_table.csv`
- `third_place_assignments.csv`
- `knockout_bracket.csv`


In [14]:
# List CSV files generated by the pipeline.
sorted(p.name for p in OUTPUT_DIR.glob("*.csv"))

['evaluation_summary.csv',
 'future_predictions.csv',
 'group_matches.csv',
 'group_table.csv',
 'knockout_bracket.csv',
 'test_predictions.csv',
 'third_place_assignments.csv']

## Streamlit app

The app reads the generated CSV files from `final_outputs_v2/` and displays the model outputs.

Run from the project folder:

```bash
python -m pip install -r requirements.txt
streamlit run app.py
```
